# raster-genie 01 — Smokehouse Creek: from COGs to a Genie answer

**One event, end to end.** In February 2024 the Smokehouse Creek fire burned more than a million acres of the Texas Panhandle. This notebook answers the three questions an insurer or emergency manager asks first — *how much land burned, how severe was it, and how many people live there* — from raster data, with SQL, and then asks the same question of a Genie Agent:

- **Land** a perimeter, the counties it touches, Sentinel-2 scenes from before and after the fire, and WorldPop population.
- **Stack** each Sentinel-2 scene into one multi-band master COG, and register every COG in a `rasters` table through a **FILE** column.
- **Query** burn severity in plain SQL: `gbx_rst_fromfile` reads only the COG blocks the perimeter needs, `gbx_rst_nbr` and `gbx_rst_mapalgebra` build the burn index, `gbx_rst_histogram` counts the severity classes.
- **Ask** the Genie Agent the same question and compare.

**Result:** acres burned by severity class, the people inside the perimeter, a severity map, and the Genie Agent's own answer.

---
_Tables in `geospatial_catalog.raster_genie`: `aois`, `rasters`, `coverage` (view)._

In [ ]:
%run ./config_nb

In [ ]:
FORCE_REBUILD = False

import time

_t_start = _t_step = time.time()
_step_no = 0


def step(name):
    """One progress line per step: its number, name, how long the previous step took and the total so far."""
    global _t_step, _step_no
    now = time.time()
    _step_no += 1
    print(f"... == step {_step_no}: {name} (previous step {now - _t_step:,.0f} s, total {now - _t_start:,.0f} s)")
    _t_step = now


## 1. Why FILE + COG

A **FILE** column holds a reference to each master COG on the Volume: the table is the catalog, nothing is copied, and Unity Catalog opens the file under the caller's grants. A **Cloud Optimized GeoTIFF** keeps its pixels in 512-pixel blocks with overviews, so `gbx_rst_fromfile` fetches only the bytes a question needs: a query over one fire reads the blocks that fire covers, not the whole file.

## 2. The fire perimeter and the counties it touches

`FirePerimeterDownloader` pulls the incident's latest perimeter from the NIFC WFIGS service (EPSG:4326 GeoJSON). The TIGER cartographic boundary files give the Texas counties; we keep the ones the perimeter touches, plus the Texas outline. Together they are the `aois` table: the named areas every raster is joined to.

In [ ]:
import json

import requests
from shapely import wkb
from shapely.ops import unary_union

step("fire perimeter and counties")

fire_dl = fire_perimeters.download(FIRE_NAME, FIRE_DIR, year=FIRE_YEAR, state=FIRE_STATE, spark=spark)
if not fire_dl.first()["is_out_file_valid"]:
    raise RuntimeError(f"No perimeter found for {FIRE_NAME} ({FIRE_STATE}, {FIRE_YEAR})")
fire_geoms = fire_perimeters.read(FIRE_DIR, FIRE_NAME, FIRE_YEAR, state=FIRE_STATE, spark=spark)

# The perimeter bbox, padded: it frames every Sentinel-2 and WorldPop download below.
_parts = [wkb.loads(bytes(r["geom_0"])) for r in fire_geoms.select("geom_0").collect()]
FIRE_POLY = unary_union(_parts)  # the perimeter as one shapely geometry (lon/lat)
_b = [g.bounds for g in _parts]
FIRE_BBOX = (min(x[0] for x in _b), min(x[1] for x in _b), max(x[2] for x in _b), max(x[3] for x in _b))
S2_BBOX = (
    FIRE_BBOX[0] - BBOX_PAD, FIRE_BBOX[1] - BBOX_PAD,
    FIRE_BBOX[2] + BBOX_PAD, FIRE_BBOX[3] + BBOX_PAD,
)
print(f"... perimeter bbox {tuple(round(v, 3) for v in FIRE_BBOX)}")
print(f"... download bbox  {tuple(round(v, 3) for v in S2_BBOX)} (+{BBOX_PAD} deg)")

In [ ]:
TIGER = "https://www2.census.gov/geo/tiger/GENZ2024/shp"
for _name in ("cb_2024_us_county_500k.zip", "cb_2024_us_state_500k.zip"):
    _dst = f"{CONTEXT_DIR}/{_name}"
    if FORCE_REBUILD or not os.path.exists(_dst):
        _resp = requests.get(f"{TIGER}/{_name}", timeout=180, stream=True)
        _resp.raise_for_status()
        with open(_dst, "wb") as fh:
            for chunk in _resp.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
        print(f"... TIGER {_name} -> {_dst}")

AOI_COLS = ["aoi_id", "name", "kind", "geom", "description"]

fire = fire_geoms.select(
    F.lit(FIRE_AOI_ID).alias("aoi_id"),
    F.lit("Smokehouse Creek fire").alias("name"),
    F.lit("fire").alias("kind"),
    F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
    F.lit("Latest NIFC WFIGS perimeter of the 2024 Smokehouse Creek fire").alias("description"),
)

tx_counties = (
    spark.read.format("shapefile_gbx").load(f"{CONTEXT_DIR}/cb_2024_us_county_500k.zip")
    .filter(F.col("STATEFP") == "48")
    .select(
        F.concat(F.lit("county_"), F.col("GEOID")).alias("aoi_id"),
        F.col("NAME").alias("name"),
        F.lit("county").alias("kind"),
        F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
        F.concat(F.col("NAME"), F.lit(" County, Texas")).alias("description"),
    )
)
# Keep the counties the perimeter touches.
touched = (
    tx_counties.alias("c")
    .join(fire.select(F.col("geom").alias("fire_geom")).alias("f"), F.expr("st_intersects(c.geom, f.fire_geom)"))
    .select(*[F.col(f"c.{c}") for c in AOI_COLS])
)
texas = (
    spark.read.format("shapefile_gbx").load(f"{CONTEXT_DIR}/cb_2024_us_state_500k.zip")
    .filter(F.col("STATEFP") == "48")
    .select(
        F.lit("tx").alias("aoi_id"),
        F.lit("Texas").alias("name"),
        F.lit("state").alias("kind"),
        F.expr("st_setsrid(st_geomfromwkb(geom_0), 4326)").alias("geom"),
        F.lit("State of Texas, TIGER cartographic boundary").alias("description"),
    )
)

aois = finalize_delta(fire.select(*AOI_COLS).unionByName(touched).unionByName(texas), "aois", do_display=False)
aois.select("aoi_id", "name", "kind", "description").orderBy("kind", "name").display()

## 3. Sentinel-2: the scenes, windowed and stacked

`StacClient.search` queries Microsoft Planetary Computer's `sentinel-2-l2a` collection over the padded perimeter bbox for two windows, before and after the fire. A scene is a candidate when its scene-level cloud cover is at most `MAX_CLOUD` percent and its no-data share is at most `MAX_NODATA` percent. That no-data figure is for the whole MGRS tile, so it is a screen for scenes cut by the satellite's orbit edge, not a measure of the perimeter; a scene with no figure at all is kept, and the coverage check on the result (section 7) is what judges it. From the candidates we pick **one scene per MGRS tile** with three rules:

- **Few dates.** The pre scenes share as few dates as possible across the tiles (the date that serves most tiles first, ties to the date closest to the fire), and so do the post scenes. Two satellite orbits cover this fire on different days, so one date is not possible: the pre scenes fall on two dates (27 January and 24 February) and the post scenes on two (10 and 12 March).
- **Same orbit.** A tile's post scene is on the same relative orbit as its pre scene wherever that orbit has a clean candidate, so the two views have the same angle and the difference between them is the ground.
- **Least cloud.** Within a tile and a day the least-cloud scene wins. A tie means the same whole percent of cloud, and then the exact figure decides.

`PRE_WINDOW` opens on 2024-01-15 rather than on 2024-02-01 because the tile in the middle of the fire (14SLE) has no clean February scene: its February scenes are cloudy or cut by the orbit edge, and its first clean one is from January. `StacClient.download` then fetches the seven bands we need (B02, B03, B04, B08, B11, B12 and the scene classification SCL) **windowed to the padded bbox**: only that part of each scene crosses the network.

In [ ]:
import sys

from pyspark.sql.types import DoubleType, StringType, StructField, StructType
from shapely.geometry import box

# choose_scene_pairs and stack_scene live in lakeflow/land/scenes.py, beside this notebook.
_lakeflow = os.path.join(os.getcwd(), "lakeflow")
if not os.path.exists(os.path.join(_lakeflow, "land", "scenes.py")):
    raise FileNotFoundError(
        f"lakeflow/land/scenes.py was not found next to this notebook ({os.getcwd()}). "
        "Fix: open this series from a Git folder, or copy the lakeflow/ folder next to the notebooks."
    )
sys.path.insert(0, _lakeflow)
from land.scenes import S2_BANDS, boa_offset_for, choose_scene_pairs, stack_scene

step("Sentinel-2 scene search")
SCENE_ASSETS = "s2_scene_assets"
S2_BAND_NAMES = ["B02", "B03", "B04", "B08", "B11", "B12", "SCL"]

# The search area: the padded bbox as a GeoJSON polygon.
_x0, _y0, _x1, _y1 = S2_BBOX
aoi_polygon = spark.createDataFrame(
    [(json.dumps({"type": "Polygon", "coordinates": [[[_x0, _y0], [_x1, _y0], [_x1, _y1], [_x0, _y1], [_x0, _y0]]]}),)],
    ["geojson"],
)


def search_window(window):
    """One window's STAC search: the hrefs of the band assets, and one candidate row per scene over the perimeter."""
    found = stac_client.search(aoi_polygon, geojson_col="geojson", collections=["sentinel-2-l2a"], datetime=window)
    items = (
        found.dropDuplicates(["item_id"])
        .select(
            "item_id", "date", "item_bbox",
            F.col("item_properties")["eo:cloud_cover"].cast("double").alias("cloud_pct"),
            F.col("item_properties")["s2:nodata_pixel_percentage"].cast("double").alias("nodata_pct"),
            F.col("item_properties")["s2:processing_baseline"].alias("baseline"),
            F.regexp_extract("item_id", r"_T(\d{2}[A-Z]{3})_", 1).alias("mgrs_tile"),
            F.regexp_extract("item_id", r"_R(\d{3})_", 1).alias("orbit"),
        )
        .collect()
    )
    rows = [r.asDict() for r in items if r["item_bbox"] and box(*r["item_bbox"]).intersects(FIRE_POLY)]
    return found, rows


if FORCE_REBUILD or not spark.catalog.tableExists(SCENE_ASSETS):
    pre_found, pre_rows = search_window(PRE_WINDOW)
    post_found, post_rows = search_window(POST_WINDOW)
    picked = choose_scene_pairs(pre_rows, post_rows, MAX_CLOUD, MAX_NODATA)
    if not picked["pre"]:
        raise RuntimeError(f"No tile has a clean pre and post scene. Fix: widen PRE_WINDOW/POST_WINDOW or raise MAX_CLOUD ({MAX_CLOUD}).")
    if picked["dropped"]:
        print(f"... no clean pre and post scene for tile(s) {', '.join(picked['dropped'])}: left out")
    if picked["orbit_mismatch"]:
        print(f"... post scene on another orbit than the pre scene for tile(s) {', '.join(picked['orbit_mismatch'])}")
    _unknown = [r["item_id"] for p in ("pre", "post") for r in picked[p].values() if r["nodata_pct"] is None]
    if _unknown:
        print(f"... {len(_unknown)} chosen scene(s) have no STAC no-data figure; the coverage check in section 7 judges them")
    meta_schema = StructType([
        StructField("item_id", StringType()), StructField("phase", StringType()), StructField("mgrs_tile", StringType()),
        StructField("date", StringType()), StructField("orbit", StringType()), StructField("cloud_pct", DoubleType()),
        StructField("nodata_pct", DoubleType()), StructField("baseline", StringType()),
    ])
    meta = spark.createDataFrame(
        [(r["item_id"], phase, r["mgrs_tile"], r["date"], r["orbit"], r["cloud_pct"], r["nodata_pct"], r["baseline"])
         for phase in ("pre", "post") for r in picked[phase].values()],
        meta_schema,
    )
    hrefs = (
        pre_found.unionByName(post_found)
        .filter(F.col("asset_name").isin(*S2_BAND_NAMES))
        .select("item_id", "asset_name", "href")
        .dropDuplicates(["item_id", "asset_name"])
    )
    finalize_delta(hrefs.join(meta, "item_id"), SCENE_ASSETS, do_display=False)
else:
    print(f"... {SCENE_ASSETS} exists; skipping the STAC search (FORCE_REBUILD=False)")

(
    spark.table(SCENE_ASSETS)
    .select("phase", "mgrs_tile", "item_id", "date", "orbit", "cloud_pct", "nodata_pct", "baseline")
    .distinct()
    .orderBy("mgrs_tile", "phase")
    .display()
)


In [ ]:
step("download the band files")
# Download the chosen bands, windowed to the padded bbox. Files that are already valid on the
# Volume are skipped, so a re-run downloads nothing new.
downloads = stac_client.download(
    spark.table(SCENE_ASSETS).select("item_id", "asset_name", "href"),
    S2_RAW_DIR,
    bbox=list(S2_BBOX),
    bbox_crs="EPSG:4326",
).select("item_id", "asset_name", "out_file_path", "out_file_sz", "is_out_file_valid").collect()

failed = [r for r in downloads if not r["is_out_file_valid"]]
print(f"... {len(downloads) - len(failed)} of {len(downloads)} band files valid in {S2_RAW_DIR}")
if failed:
    raise RuntimeError(f"{len(failed)} band downloads failed, e.g. {failed[0]['item_id']} {failed[0]['asset_name']}")

Now stack each scene. `stack_scene` (in `lakeflow/land/scenes.py`) writes **one master COG per scene**: the seven bands on the 10 m grid of B04 (the 20 m bands B11, B12 and SCL are resampled, bilinear for reflectance and nearest for SCL), 512-pixel blocks, overviews at 2/4/8/16, GeoBrix's own compression options, nodata 0, and the band names as band descriptions. The overviews are **nearest neighbour**, not averages: a COG has one overview resampling for all its bands, and SCL holds class ids, which an average would blend into numbers that mean nothing. With nearest overviews a coarse read can still be masked with SCL. Reflectance has Sentinel-2's +1000 processing offset removed first, so the indices are taken on true reflectance. The file carries no internal mask band: nodata is the only validity signal, which is what the Unity Catalog windowed reader plans for.

In [ ]:
import datetime

import rasterio

step("stack the master COGs")

band_files = {}
for r in downloads:
    band_files.setdefault(r["item_id"], {})[r["asset_name"]] = r["out_file_path"]

scene_rows = (
    spark.table(SCENE_ASSETS)
    .select("item_id", "phase", "mgrs_tile", "date", "cloud_pct", "baseline")
    .distinct()
    .orderBy("phase", "mgrs_tile")
    .collect()
)

masters = []  # one record per scene master COG
for s in scene_rows:
    day = s["date"][:10]
    asset_id = f"s2_{s['mgrs_tile']}_{day.replace('-', '')}"
    out = f"{S2_MASTER_DIR}/{asset_id}.tif"
    if FORCE_REBUILD or not os.path.exists(out):
        stack_scene(band_files[s["item_id"]], out, bands=S2_BANDS, ref="B04", boa_offset=boa_offset_for(s["baseline"]))
        print(f"... stacked {asset_id} ({s['phase']}, cloud {s['cloud_pct']:.1f}%)")
    else:
        print(f"... {asset_id} exists (skip; FORCE_REBUILD=False)")
    with rasterio.open(out) as ds:
        km2 = ds.width * ds.height * abs(ds.res[0] * ds.res[1]) / 1e6
    masters.append(
        dict(asset_id=asset_id, day=datetime.date.fromisoformat(day), cloud_pct=float(s["cloud_pct"]),
             path=out, bytes=os.path.getsize(out), km2=km2)
    )
print(f"... {len(masters)} Sentinel-2 master COGs in {S2_MASTER_DIR}")

## 4. WorldPop population

WorldPop's 2020 constrained population grid for the USA is a single national GeoTIFF. `WorldPopDownloader` windows it to the same padded bbox; where the server cannot serve byte ranges it fetches the national file once into `POP_CACHE_DIR` first, and every later call windows that local copy. `prepare_cogs` rewrites the window as a master COG in `population/`: the same layout as the Sentinel-2 scenes, so one table catalogs both.

In [ ]:
step("WorldPop population")
pop_dl = worldpop.download(S2_BBOX, POP_RAW_DIR, spark=spark, cache_dir=POP_CACHE_DIR).first()
if not pop_dl["is_out_file_valid"]:
    raise RuntimeError("WorldPop window does not overlap the download bbox")
pop_summary = prepare_cogs(
    pop_dl["out_file_path"], POP_DIR, blocksize=512, compress="auto", bigtiff="IF_SAFER", skip_if_exists=not FORCE_REBUILD
)
pop_cog = pop_summary["results"][0]["output_path"]
if pop_cog is None:
    raise RuntimeError(f"population COG failed: {pop_summary['results'][0]['status']}")
print(f"... population master COG: {pop_cog} ({os.path.getsize(pop_cog) / 1e6:,.1f} MB)")

## 5. Land the COGs as a FILE table

One row per master COG: its product, tier, area, date, band order, source and licence, and the **FILE EXTERNAL** column `file` that points at the COG on the Volume. All of these are marked **enhanced**, the tier for focus areas and events. We read the technical metadata from the COG itself with SQL before writing: a header-only tile gives the CRS, pixel size and bounding box (`gbx_rst_srid`, `gbx_rst_pixelheight`, `gbx_rst_boundingbox`, reprojected to a `GEOMETRY(4326)` footprint with `st_transform`), and `valid_pct` is the share of data pixels read from a coarse overview, so landing never decodes a full scene.

In [ ]:
step("land the COGs as a FILE table")
RASTERS = f"{catalog_name}.{schema_name}.rasters"

if FORCE_REBUILD or not spark.catalog.tableExists(RASTERS):
    s2_source = "Sentinel-2 L2A, Microsoft Planetary Computer"
    s2_license = "Copernicus Sentinel data: free, full and open"
    rows = [
        (m["asset_id"], "s2_l2a", "enhanced", FIRE_AOI_ID, m["day"], list(S2_BANDS),
         s2_source, s2_license, m["cloud_pct"], m["path"], 160.0)
        for m in masters
    ]
    rows.append(
        ("pop_worldpop_2020_" + FIRE_AOI_ID, "population", "enhanced", FIRE_AOI_ID, datetime.date(2020, 1, 1),
         ["population"], "WorldPop 2020 constrained population, USA (people per pixel)", "CC BY 4.0",
         None, pop_cog, None)
    )
    landing_schema = StructType([
        StructField("asset_id", StringType()), StructField("product", StringType()),
        StructField("tier", StringType()), StructField("aoi_id", StringType()),
        StructField("acquired_date", DateType()), StructField("bands", ArrayType(StringType())),
        StructField("source", StringType()), StructField("license", StringType()),
        StructField("cloud_pct", DoubleType()), StructField("path", StringType()),
        StructField("preview_res", DoubleType()),  # pixel size of the coarse overview read for valid_pct
    ])
    landing = spark.createDataFrame(rows, landing_schema)

    # t: a header-only tile (CRS, pixel size, bounds). preview: a coarse overview read (valid pixels).
    (
        landing
        .withColumn("t", rx.rst_fromfile(F.col("path")))
        .withColumn("preview", F.expr("gbx_rst_fromfile(path, 'GTiff', NULL, NULL, preview_res)"))
        .createOrReplaceTempView("landing_tiles")
    )
    rasters_df = spark.sql("""
        SELECT asset_id, product, tier, aoi_id, acquired_date, bands,
               concat('EPSG:', CAST(gbx_rst_srid(t) AS STRING)) AS crs,
               CASE WHEN gbx_rst_srid(t) = 4326
                    THEN abs(gbx_rst_pixelheight(t)) * 111320.0
                    ELSE abs(gbx_rst_pixelwidth(t)) END AS resolution_m,
               st_transform(st_setsrid(st_geomfromwkb(gbx_rst_boundingbox(t)), gbx_rst_srid(t)), 4326) AS footprint,
               round(100.0 * gbx_rst_pixelcount(preview)[0] / (gbx_rst_width(preview) * gbx_rst_height(preview)), 1) AS valid_pct,
               cloud_pct, source, license, path
        FROM landing_tiles
    """)
    gbx_file_write(rasters_df, RASTERS, file_mode="external", file_col="file", overwrite=True)
else:
    print(f"... {RASTERS} exists (skip; FORCE_REBUILD=False)")

spark.table(RASTERS).select(
    "asset_id", "product", "tier", "acquired_date", "bands", "crs", "resolution_m", "valid_pct", "cloud_pct", "file"
).orderBy("product", "acquired_date").display()

In [ ]:
# Genie reads the comments: describe the tables and their columns.
def comment_table(tbl, table_comment, columns):
    spark.sql(f"COMMENT ON TABLE {tbl} IS '{table_comment}'")
    for col, text in columns.items():
        spark.sql(f"ALTER TABLE {tbl} ALTER COLUMN {col} COMMENT '{text}'")


comment_table(RASTERS, "One row per master COG: Sentinel-2 L2A scenes and WorldPop population, each registered through a FILE column.", {
    "asset_id": "Unique id of the master COG",
    "product": "s2_l2a (Sentinel-2 L2A), dem, land_cover or population",
    "tier": "standard (all of Texas) or enhanced (focus areas and events)",
    "aoi_id": "The area (aois.aoi_id) the asset was landed for",
    "acquired_date": "Acquisition date of the scene; the reference year for population",
    "bands": "Band names in band order; the position in this array is the 1-based band index",
    "crs": "Coordinate reference system of the COG",
    "resolution_m": "Native pixel size in metres",
    "footprint": "Extent of the COG as a GEOMETRY(4326)",
    "valid_pct": "Percent of pixels that hold data (not nodata)",
    "cloud_pct": "Scene-level cloud cover in percent (Sentinel-2 only)",
    "source": "Where the data comes from",
    "license": "Licence of the data",
    "path": "Volume path of the COG",
    "file": "FILE EXTERNAL reference to the COG; pass it to gbx_rst_fromfile",
})
comment_table(f"{catalog_name}.{schema_name}.aois", "Named areas: the fire perimeter, the Texas counties it touches and the Texas outline.", {
    "aoi_id": "Unique id of the area",
    "name": "Display name",
    "kind": "fire, county or state",
    "geom": "Boundary as a GEOMETRY(4326)",
    "description": "What the area is",
})

In [ ]:
# What landing produced, and what that means at scale.
s2_gb = sum(m["bytes"] for m in masters) / 1e9
s2_km2 = sum(m["km2"] for m in masters)
n_assets = spark.table(RASTERS).count()
print(f"... {n_assets} assets in {RASTERS}: {len(masters)} Sentinel-2 scenes ({s2_gb:,.2f} GB over about {s2_km2:,.0f} km2) + 1 population COG")
print(
    f"... at this density, one Sentinel-2 pass over all of Texas (695,662 km2) is about {s2_gb / s2_km2 * 695_662 / 1e3:,.1f} TB "
    "of master COGs, and a year of quarterly passes about four times that: the same table, the same query."
)

## 6. The `coverage` view

What do we hold, where, at what resolution and how recent? `coverage` answers that per product, tier and area by joining each COG's footprint to the `aois`. A data-product owner can ask it the first question of any engagement.

In [ ]:
step("coverage view")
spark.sql("""
    CREATE OR REPLACE VIEW coverage
    COMMENT 'What raster data we hold: one row per product, tier and area, with the best resolution, date range, asset count and mean valid percent.'
    AS
    SELECT r.product, r.tier, a.aoi_id, a.name AS aoi_name, a.kind AS aoi_kind,
           min(r.resolution_m) AS best_resolution_m,
           min(r.acquired_date) AS first_date,
           max(r.acquired_date) AS last_date,
           count(*) AS assets,
           round(avg(r.valid_pct), 1) AS mean_valid_pct
    FROM rasters r
    JOIN aois a ON st_intersects(r.footprint, a.geom)
    GROUP BY r.product, r.tier, a.aoi_id, a.name, a.kind
""")
spark.table("coverage").orderBy("aoi_kind", "aoi_name", "product").display()

## 7. Burn severity in plain SQL

The classes follow the standard dNBR table (Key & Benson, 2006, *FIREMON*, USDA Forest Service RMRS-GTR-164, as published by USGS):

| dNBR | Class | Severity |
|---|---|---|
| below 0.1 | 1 | Unburned |
| 0.1 to 0.27 | 2 | Low |
| 0.27 to 0.44 | 3 | Moderate-low |
| 0.44 to 0.66 | 4 | Moderate-high |
| 0.66 and above | 5 | High |

Key & Benson also list two regrowth classes below -0.1 (enhanced regrowth, high and low); they are folded into class 1, Unburned, here. The table was written for forest burns. These are **winter scenes** of a grass and brush landscape: the sun is low, the grass is dormant, and a grass fire changes NBR far less than a forest fire does. So most of this fire reads as Unburned or Low, and the classes say more about relative severity than about absolute fuel loss.

The query is one SQL statement over the `rasters`, `s2_scene_assets` and `aois` tables:

1. **Pair** each MGRS tile's pre and post scene whose footprints both meet the perimeter (the tile is part of the `asset_id`; the relative orbit comes from `s2_scene_assets`).
2. **Read** only the perimeter from each scene with `gbx_rst_fromfile(file, 'GTiff', <perimeter>, 'EPSG:4326', <resolution>)`: the geometry clips the window and the resolution picks the overview, so only the COG blocks the perimeter touches are fetched.
3. **Index** each read with `gbx_rst_nbr(tile, nir, swir)`, the normalized burn ratio `(NIR - SWIR) / (NIR + SWIR)`, and take the **SCL band** of each read with `gbx_rst_band`. The band numbers are looked up from the `bands` column: NIR is B08, SWIR is B12.
4. **Difference and mask** them in one `gbx_rst_mapalgebra`: dNBR = pre NBR - post NBR, left as NoData wherever either scene's SCL class is no data (0), saturated (1), cloud shadow (3), cloud (8, 9), thin cirrus (10) or snow (11). The masked share is reported below.
5. **Classify** dNBR into the five classes with a second `gbx_rst_mapalgebra`, and **merge** the tiles with `gbx_rst_merge`, so the overlap between neighbouring MGRS tiles is counted once.
6. **Count** the pixels per class with `gbx_rst_histogram`, and turn pixels into area with the tile's own pixel size. `pct_of_valid_pixels` is each class's share of the pixels that hold data and are not masked inside the perimeter.

The answer is saved as `burn_severity_summary`, so a re-run does not read the COGs again. A cell after the query checks that the unmasked pixels cover at least `MIN_COVER` of the perimeter's area, so a scene that is cut by the orbit edge, or one whose no-data figure STAC does not give, cannot pass unnoticed.

The window one `gbx_rst_fromfile` call returns must fit the **64 MiB** a Unity Catalog function can return. Before the query we size it: the perimeter's bounding box in the scene CRS, divided into pixels at the read resolution, times the seven bands at two bytes each. If that is over 64 MiB the read steps up to the next coarser overview of the COG (20, 40, 80, then 160 m), and the cell says which one it chose.

In [ ]:
import math

from pyproj import Transformer

FROMFILE_CAP_BYTES = 64 * 1024**2
NATIVE_M = 10.0
LEVELS_M = [NATIVE_M * f for f in (1, 2, 4, 8, 16)]  # pixel sizes of the master COG and its overviews

scene_crs = spark.table(RASTERS).filter("product = 's2_l2a'").select("crs").first()["crs"]
_to_scene = Transformer.from_crs("EPSG:4326", scene_crs, always_xy=True)
_xy = [_to_scene.transform(x, y) for x in (FIRE_BBOX[0], FIRE_BBOX[2]) for y in (FIRE_BBOX[1], FIRE_BBOX[3])]
extent_x = max(p[0] for p in _xy) - min(p[0] for p in _xy)
extent_y = max(p[1] for p in _xy) - min(p[1] for p in _xy)


def window_bytes(res_m):
    """Decoded size of the perimeter's window at a pixel size: pixels x bands x 2 bytes."""
    return math.ceil(extent_x / res_m) * math.ceil(extent_y / res_m) * len(S2_BANDS) * 2


# gbx_rst_fromfile reads the coarsest overview at least as fine as the resolution asked for.
burn_res_m = max([m for m in LEVELS_M if m <= BURN_RES_M] or [NATIVE_M])
print(f"... asked for {BURN_RES_M:g} m: the read uses the {burn_res_m:g} m overview, "
      f"a window of {window_bytes(burn_res_m) / 1024**2:,.1f} MiB (limit {FROMFILE_CAP_BYTES / 1024**2:.0f} MiB)")
while window_bytes(burn_res_m) > FROMFILE_CAP_BYTES:
    coarser = [m for m in LEVELS_M if m > burn_res_m]
    if not coarser:
        raise RuntimeError("The perimeter window is over 64 MiB at every overview. Fix: query a smaller part of the perimeter.")
    print(f"... {burn_res_m:g} m is over the limit, so the read steps up to the {coarser[0]:g} m overview")
    burn_res_m = coarser[0]
print(f"... burn severity is read at {burn_res_m:g} m ({window_bytes(burn_res_m) / 1024**2:,.1f} MiB per window)")
map_res_m = max(MAP_RES_M, burn_res_m)

In [ ]:
step("burn severity query")
MIN_COVER = 0.95  # the unmasked pixels must cover at least this share of the perimeter's area
SCL_MASKED = (0, 1, 3, 8, 9, 10, 11)  # no data, saturated, cloud shadow, cloud (medium, high), thin cirrus, snow
_bad = " || ".join(f"{var} == {c}" for var in ("C", "D") for c in SCL_MASKED)
MASK_EXPR = f"({_bad}) ? -9999 : (A - B)"  # C and D are the SCL bands of the pre and post reads; -9999 is NoData

BURN_HEAD = """
WITH perimeter AS (
  SELECT geom FROM aois WHERE aoi_id = '{fire_id}'
),
meta AS (
  SELECT DISTINCT mgrs_tile, to_date(substr(date, 1, 10)) AS d, regexp_extract(item_id, '_R([0-9]+)_', 1) AS orbit
  FROM s2_scene_assets
),
scenes AS (
  SELECT r.asset_id, m.mgrs_tile, m.orbit, r.acquired_date, r.bands, r.file, r.footprint
  FROM rasters r
  JOIN meta m ON r.asset_id = concat('s2_', m.mgrs_tile, '_', date_format(m.d, 'yyyyMMdd'))
  WHERE r.product = 's2_l2a' AND r.aoi_id = '{fire_id}'
),
pairs AS (
  SELECT pre.mgrs_tile, pre.acquired_date AS pre_date, pre.orbit AS pre_orbit,
         post.acquired_date AS post_date, post.orbit AS post_orbit,
         pre.bands, pre.file AS pre_file, post.file AS post_file, pre.footprint AS fp, p.geom
  FROM scenes pre
  JOIN scenes post ON post.mgrs_tile = pre.mgrs_tile
  CROSS JOIN perimeter p
  WHERE pre.acquired_date  BETWEEN DATE'{pre_from}'  AND DATE'{pre_to}'
    AND post.acquired_date BETWEEN DATE'{post_from}' AND DATE'{post_to}'
    AND st_intersects(pre.footprint, p.geom)
    AND st_intersects(post.footprint, p.geom)
),
reads AS (
  SELECT mgrs_tile, pre_date, pre_orbit, post_date, post_orbit,
         CAST(array_position(bands, 'B08') AS INT) AS nir,
         CAST(array_position(bands, 'B12') AS INT) AS swir,
         CAST(array_position(bands, 'SCL') AS INT) AS scl,
         gbx_rst_fromfile(pre_file,  'GTiff', geom, 'EPSG:4326', {res}) AS pre_tile,
         gbx_rst_fromfile(post_file, 'GTiff', geom, 'EPSG:4326', {res}) AS post_tile
  FROM pairs
),
parts AS (
  SELECT mgrs_tile, pre_date, pre_orbit, post_date, post_orbit,
         gbx_rst_nbr(pre_tile,  nir, swir) AS pre_nbr,
         gbx_rst_nbr(post_tile, nir, swir) AS post_nbr,
         gbx_rst_band(pre_tile,  scl) AS pre_scl,
         gbx_rst_band(post_tile, scl) AS post_scl
  FROM reads
  WHERE pre_tile IS NOT NULL AND post_tile IS NOT NULL
),
dnbr AS (
  SELECT mgrs_tile, pre_date, pre_orbit, post_date, post_orbit, pre_nbr, post_nbr,
         gbx_rst_mapalgebra(array(pre_nbr, post_nbr, pre_scl, post_scl), '{mask_expr}') AS tile,
         gbx_rst_mapalgebra(array(pre_nbr, post_nbr), 'A - B') AS unmasked
  FROM parts
)
"""

BURN_CTES = BURN_HEAD + """,
severity AS (
  SELECT gbx_rst_merge(collect_list(
           gbx_rst_mapalgebra(array(tile), 'A < 0.1 ? 1 : (A < 0.27 ? 2 : (A < 0.44 ? 3 : (A < 0.66 ? 4 : 5)))')
         )) AS tile
  FROM dnbr
)
"""

BURN_AREA = """
SELECT pos + 1 AS class_id,
       CASE pos + 1 WHEN 1 THEN 'Unburned' WHEN 2 THEN 'Low' WHEN 3 THEN 'Moderate-low'
                    WHEN 4 THEN 'Moderate-high' ELSE 'High' END AS severity,
       n AS pixels,
       round(n * abs(gbx_rst_pixelwidth(tile) * gbx_rst_pixelheight(tile)) / 1e6, 1) AS km2,
       round(n * abs(gbx_rst_pixelwidth(tile) * gbx_rst_pixelheight(tile)) / 4046.8564224, 0) AS acres,
       round(100.0 * n / sum(n) OVER (), 1) AS pct_of_valid_pixels
FROM (SELECT tile, gbx_rst_histogram(tile, 5, 0.5, 5.5, false)['band_1'] AS counts FROM severity)
LATERAL VIEW posexplode(counts) h AS pos, n
ORDER BY class_id
"""

# One row per MGRS tile: the scenes it used, how much of it the SCL mask removed, and its mean dNBR.
BURN_TILES = """
SELECT mgrs_tile, pre_date, pre_orbit, post_date, post_orbit,
       gbx_rst_pixelcount(unmasked)[0] AS px_data,
       gbx_rst_pixelcount(tile)[0] AS px_clear,
       round(100.0 * (1 - gbx_rst_pixelcount(tile)[0] / gbx_rst_pixelcount(unmasked)[0]), 1) AS masked_pct,
       round(gbx_rst_avg(pre_nbr)[0], 3) AS mean_pre_nbr,
       round(gbx_rst_avg(post_nbr)[0], 3) AS mean_post_nbr,
       round(gbx_rst_avg(tile)[0], 3) AS mean_dnbr
FROM dnbr
ORDER BY mgrs_tile
"""

# Where two neighbouring tiles overlap inside the perimeter they see the same ground: compare their scenes there.
BURN_OVERLAP = """
, overlaps AS (
  SELECT a.mgrs_tile AS tile_a, b.mgrs_tile AS tile_b, a.pre_date AS a_pre_date, a.pre_orbit AS a_pre_orbit,
         b.pre_date AS b_pre_date, b.pre_orbit AS b_pre_orbit,
         CAST(array_position(a.bands, 'B08') AS INT) AS nir, CAST(array_position(a.bands, 'B12') AS INT) AS swir,
         a.pre_file AS a_pre_file, a.post_file AS a_post_file, b.pre_file AS b_pre_file, b.post_file AS b_post_file,
         st_intersection(st_intersection(a.fp, b.fp), a.geom) AS og
  FROM pairs a JOIN pairs b ON a.mgrs_tile < b.mgrs_tile
  WHERE st_intersects(a.fp, b.fp)
),
nbrs AS (
  SELECT tile_a, tile_b, a_pre_date, a_pre_orbit, b_pre_date, b_pre_orbit,
         gbx_rst_avg(gbx_rst_nbr(gbx_rst_fromfile(a_pre_file,  'GTiff', og, 'EPSG:4326', {res}), nir, swir))[0] AS a_pre,
         gbx_rst_avg(gbx_rst_nbr(gbx_rst_fromfile(b_pre_file,  'GTiff', og, 'EPSG:4326', {res}), nir, swir))[0] AS b_pre,
         gbx_rst_avg(gbx_rst_nbr(gbx_rst_fromfile(a_post_file, 'GTiff', og, 'EPSG:4326', {res}), nir, swir))[0] AS a_post,
         gbx_rst_avg(gbx_rst_nbr(gbx_rst_fromfile(b_post_file, 'GTiff', og, 'EPSG:4326', {res}), nir, swir))[0] AS b_post
  FROM overlaps
  WHERE st_area(og) > 0.001  -- square degrees: about 10 km2 or more
)
SELECT tile_a, tile_b, a_pre_date, a_pre_orbit, b_pre_date, b_pre_orbit,
       round(a_pre, 3) AS a_pre_nbr, round(b_pre, 3) AS b_pre_nbr, round(a_pre - b_pre, 3) AS pre_diff,
       round(a_post, 3) AS a_post_nbr, round(b_post, 3) AS b_post_nbr, round(a_post - b_post, 3) AS post_diff,
       round(a_pre - a_post, 3) AS a_dnbr, round(b_pre - b_post, 3) AS b_dnbr
FROM nbrs
ORDER BY tile_a, tile_b
"""

burn_params = dict(
    fire_id=FIRE_AOI_ID, mask_expr=MASK_EXPR,
    pre_from=PRE_WINDOW.split("/")[0], pre_to=PRE_WINDOW.split("/")[1],
    post_from=POST_WINDOW.split("/")[0], post_to=POST_WINDOW.split("/")[1],
)
# The query reads the COGs, so run it once and keep the answer.
burn_area = finalize_delta(
    spark.sql(BURN_CTES.format(res=burn_res_m, **burn_params) + BURN_AREA), "burn_severity_summary", do_display=False
)
if burn_area.count() == 0:
    raise RuntimeError("No pre and post scene pair of one MGRS tile meets the perimeter; check the windows and MAX_CLOUD")
burn_area.display()

# Coverage check: the unmasked pixels against the perimeter's area (equal-area projection).
perimeter_acres = spark.sql(
    f"SELECT st_area(st_transform(geom, 5070)) / 4046.8564224 FROM aois WHERE aoi_id = '{FIRE_AOI_ID}'"
).first()[0]
covered_acres = burn_area.agg(F.sum("acres")).first()[0]
print(f"... the unmasked pixels cover {covered_acres:,.0f} of the perimeter's {perimeter_acres:,.0f} acres "
      f"({100 * covered_acres / perimeter_acres:.1f}%)")
if covered_acres < MIN_COVER * perimeter_acres:
    raise RuntimeError(
        f"The scenes cover only {100 * covered_acres / perimeter_acres:.1f}% of the perimeter (limit {100 * MIN_COVER:.0f}%). "
        "Fix: lower MAX_NODATA, widen the windows or pick other scenes (see the table in section 3)."
    )


### Do the tiles agree?

Each MGRS tile is its own pair of scenes, so a difference between neighbouring tiles can come from the ground or from the scenes. The first table gives, per tile, the pre and post dates and relative orbits, the share of the tile's pixels the SCL mask removed (`masked_pct`), the mean pre and post NBR, and the mean dNBR. The second table settles the question where two tiles overlap: inside the perimeter they see **the same ground**, so a difference there is the scenes, not the fire. `post_diff` near zero and `pre_diff` of several hundredths means the two **pre** scenes disagree about the ground before the fire, and the dNBR of one tile is shifted against the other's by that amount. Read both tables before reading the map.

In [ ]:
burn_tiles = spark.sql(BURN_HEAD.format(res=burn_res_m, **burn_params) + BURN_TILES)
burn_tiles.display()
_t = [r.asDict() for r in burn_tiles.collect()]
_data, _clear = sum(r["px_data"] for r in _t), sum(r["px_clear"] for r in _t)
print(f"... the SCL mask removed {100 * (1 - _clear / _data):.1f}% of the tile pixels in the perimeter (cloud, shadow, snow, saturated, no data)")
_mean = sum(r["mean_dnbr"] * r["px_clear"] for r in _t) / _clear
for r in _t:
    if abs(r["mean_dnbr"] - _mean) > 0.05 and r["px_clear"] > 0.05 * _clear:
        print(f"... NOTE tile {r['mgrs_tile']}: mean dNBR {r['mean_dnbr']:.3f} against {_mean:.3f} over the whole perimeter "
              f"(pre {r['pre_date']} orbit {r['pre_orbit']}, post {r['post_date']} orbit {r['post_orbit']})")

# The same ground seen by two tiles.
overlap = spark.sql(BURN_HEAD.format(res=burn_res_m, **burn_params) + BURN_OVERLAP.format(res=burn_res_m))
overlap.display()
for r in overlap.collect():
    verdict = "the scenes agree"
    if abs(r["pre_diff"]) > 0.03 or abs(r["post_diff"]) > 0.03:
        verdict = (f"the scenes DISAGREE over the same ground (pre {r['pre_diff']:+.3f}, post {r['post_diff']:+.3f}): "
                   f"dNBR is {r['a_dnbr']:.3f} from {r['tile_a']} and {r['b_dnbr']:.3f} from {r['tile_b']}, and the severity classes either side of the tile edge are not comparable")
    print(f"... overlap {r['tile_a']} (pre {r['a_pre_date']} orbit {r['a_pre_orbit']}) with {r['tile_b']} (pre {r['b_pre_date']} orbit {r['b_pre_orbit']}): {verdict}")


**What the tables show for this fire.** 14SLE and 14SME agree: same orbit, same dates, the same dNBR over the ground they share. 14SKE and 14SLE do not. Over the same ground their **post** scenes agree (mean NBR within about 0.01) but their **pre** scenes differ by about 0.08: 14SKE's pre scene (24 February, orbit 055) reads lower than 14SLE's (27 January, orbit 012). So the dNBR over the west lobe (14SKE) comes out about 0.09 lower than over the middle of the fire (14SLE), and the break at their tile edge on the map below is the scenes, not the fire. 14SLE has no clean February scene, so its pre scene cannot be moved to a date near 14SKE's, and this data cannot say which of the two pre scenes is nearer the truth. Read the west lobe's Unburned class as *a low dNBR against one pre scene*, not as *did not burn*.

Now the people. The WorldPop COG is read the same way, clipped to the perimeter, and the population inside is the pixel mean times the count of valid pixels.

In [ ]:
step("people inside the perimeter")
people_in_perimeter = spark.sql(f"""
    SELECT round(avg_values[0] * valid_counts[0]) AS people_in_perimeter
    FROM (
      SELECT gbx_rst_avg(t) AS avg_values, gbx_rst_pixelcount(t) AS valid_counts
      FROM (
        SELECT gbx_rst_fromfile(r.file, 'GTiff', a.geom, 'EPSG:4326') AS t
        FROM rasters r CROSS JOIN aois a
        WHERE r.product = 'population' AND a.aoi_id = '{FIRE_AOI_ID}'
          AND st_intersects(r.footprint, a.geom)
      )
    )
""").first()[0]

_acres = burn_area.agg(F.sum("acres")).first()[0]
_burned = burn_area.filter("class_id >= 2").agg(F.sum("acres")).first()[0]
print(f"... {_acres:,.0f} acres inside the perimeter, {_burned:,.0f} of them at low severity or worse")
if people_in_perimeter is None:
    print("... no WorldPop pixel with data falls inside the perimeter; check the population COG and its footprint")
else:
    print(f"... about {people_in_perimeter:,.0f} people live inside the perimeter (WorldPop 2020)")

## 8. The map

The same query, read at the coarser `MAP_RES_M` (or the severity resolution, if that is coarser) so the figure stays light, gives the merged severity-class raster. `vizx.plot_tile` drapes it over an Esri basemap, with the NIFC perimeter outlined.

In [ ]:
step("map")
import geopandas as gpd
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

SEVERITY_CLASSES = [
    ("Unburned", "#a6d96a"), ("Low", "#ffffbf"), ("Moderate-low", "#fdae61"),
    ("Moderate-high", "#f46d43"), ("High", "#a50026"),
]
severity_cmap = ListedColormap([color for _, color in SEVERITY_CLASSES])
# Class k always takes colour k, whichever classes the raster holds.
severity_norm = BoundaryNorm(np.arange(0.5, 6.5, 1.0), severity_cmap.N)

severity_row = spark.sql(
    BURN_CTES.format(res=map_res_m, **burn_params)
    + "SELECT tile, gbx_rst_srid(tile) AS srid FROM severity"
).first()
fire_wkt = spark.sql(f"SELECT st_astext(geom) AS wkt FROM aois WHERE aoi_id = '{FIRE_AOI_ID}'").first()["wkt"]

ax = plot_tile(
    severity_row["tile"],
    band=1,
    cmap=severity_cmap,
    basemap_source="topo",
    title="Smokehouse Creek fire: dNBR burn severity inside the NIFC perimeter",
)
next(im for im in ax.images if im.get_zorder() == max(i.get_zorder() for i in ax.images)).set_norm(severity_norm)
ax.legend(
    handles=[Patch(facecolor=color, edgecolor="grey", label=f"{k}  {name}") for k, (name, color) in enumerate(SEVERITY_CLASSES, start=1)],
    title="dNBR severity class", loc="lower left",
)
gpd.GeoSeries.from_wkt([fire_wkt], crs=4326).to_crs(int(severity_row["srid"])).boundary.plot(
    ax=ax, color="black", linewidth=1.2
)

## 9. Ask the Genie Agent

The SQL above is what a person writes when they know the functions. A **Genie Agent** is given the same tables, the `gbx_rst_*` functions and a short set of instructions, and composes the query itself from a plain-English question. The call below uses Agent mode and shows the SQL the agent wrote next to its answer, so the two approaches can be compared. It needs the id of the raster-genie Genie space in `GENIE_SPACE_ID` (set in `config_nb`). The SQL above counts the people inside the whole fire perimeter, burned or not, so compare the agent's population figure on that basis.

In [ ]:
step("Genie Agent")
Q3 = "How much land burned in the Smokehouse Creek fire, how severe, and how many people live in the burned area?"

if not GENIE_SPACE_ID:
    print("... GENIE_SPACE_ID is empty in config_nb; skipping the Genie Agent step.")
    print("... Set it to the id of the raster-genie Genie space and run this cell again.")
else:
    answer = ask_genie(GENIE_SPACE_ID, Q3)
    print(f"Question: {Q3}")
    print(f"Status:   {answer['status']} in {answer['seconds']} s")
    if answer["error"]:
        print(f"Error:    {answer['error']}")
    for i, statement in enumerate(answer["sql"], start=1):
        print(f"\n-- SQL the agent ran ({i} of {len(answer['sql'])})\n{statement}")
    print("\nAnswer:")
    for text in answer["text"]:
        print(text)
step("done")


## What we built

- **`aois`** (Delta): the Smokehouse Creek perimeter, the Texas counties it touches and the Texas outline.
- **`rasters`** (Delta, FILE EXTERNAL): one row per master COG, with its footprint, CRS, resolution, valid percent, band order, source and licence.
- **`coverage`** (view): what we hold per product, tier and area.
- **`burn_severity_summary`** (Delta): the severity query's answer, kept so the notebook does not read the COGs twice.
- The **burn severity** by class in acres and square kilometres, the **people** inside the perimeter, and a severity map.
- The Genie Agent's SQL and answer to the same question.

GeoBrix: `StacClient`, `FirePerimeterDownloader`, `WorldPopDownloader`, `prepare_cogs`, `gbx_file_write`, `gbx_rst_fromfile`, `gbx_rst_nbr`, `gbx_rst_mapalgebra`, `gbx_rst_merge`, `gbx_rst_histogram`, `gbx_rst_avg`, `gbx_rst_pixelcount`, `genie.ask`, `vizx.plot_tile`.